# SWIFT-MAN (Taylor's Version): train an 8-bit pop star to play Ms. Pac-Man

**Choose three numbers. Run the notebook. Watch a digital Taylor Swift learn, badly at first, to recruit Swifties and secure the bag.**

Under all the glitter, this is a real **Deep Q-Network (DQN)**, which is reinforcement learning's version of "learn from your mistakes, but with math."
The agent sees the Atari screen, picks a joystick move, and gets points from the game. A neural network slowly figures out which moves lead to more points.

What you watch and what the AI sees are **two very different things**:

| | The neural network sees | You see |
|---|---|---|
| Hero | a gray smudge | an 8-bit Taylor Swift, Mario-style, red lip and all |
| Pellets | slightly lighter gray smudges | tiny Swifties and bags of money |
| Ghosts | darker gray smudges | HATERS, PAPARAZZI, TICKET BOTS, CRITICS |
| The maze | 84 × 84 pixels of beige despair | a rainbow plasma fever dream with CRT glitches |

The makeover only touches the recorded GIFs. Training is identical to the original classroom notebook. It sees exactly the same screens, gets the same rewards, and learns with the same math. Yes, we checked.

1. Open this notebook in Jupyter, VS Code, or Google Colab. In Colab, pick **Runtime → Change runtime type → T4 GPU** if you're feeling fancy.
2. Choose your three settings in the next cell.
3. Choose **Run All** (Colab: **Runtime → Run all**). Setup runs itself. Locally, gameplay clips pop up in a floating player at **4× speed**; a copy also appears inline.
4. Compare the five before/after scores and save your results, then reflect on them with the dignity of an artist in her reflective era.

If you're using Claude Code or Codex, ask it to run this notebook. The code tells it to ask you for your three choices before training.

A short run checks that everything works. Learning actual Atari skill takes far longer. More episodes do not guarantee better play, much like more re-recordings do not guarantee a better song (they usually do, though).

> **Parody disclaimer:** SWIFT-MAN is a fan-made classroom joke. It is not affiliated with, endorsed by, or known to Taylor Swift, her team, her label, or Atari.
> No Swifties were harmed; they were *recruited*. No real money was secured. The net worth counter is a bit.
> Built with Claude Code on top of the course's original DQN notebook (`pepealonso95/pacman-dqn`).

## 1. your three choices (the only three decisions you get to make, so make them count)

| Setting | What it actually means | Starting point |
|---|---|---|
| **Exploration** | Fraction of training moves chosen at random. `0.20` means 20% of the time Taylor wanders off to "explore her sound." | `0.20` |
| **Episodes** | How many games to train on. Each game ends at game over or a fixed time limit (like a stadium curfew). | `100` |
| **Learning rate** | How much the network changes after each lesson. Too big and it rewrites its whole personality every update; too small and it barely learns. | `0.0001` |

Change only the three numbers below. Try `5` episodes first as a sound check, then do the full tour.
Exploration stays **constant** after warm-up, so its meaning is easy to compare across runs.
Before any learning, a short warm-up of 1,000 purely random moves fills the memory (the opening act).

In [ ]:
# AI ASSISTANT INSTRUCTION:
# Before running training, ask the student:
# "Which exploration rate (0 to 1), number of episodes (positive whole number),
# and learning rate (greater than 0 and at most 1) do you want to use?"
# Explain the three choices briefly. Wait for their answer, then edit ONLY
# these three values. If they already chose values in this conversation,
# use those choices without asking again. Do not silently choose for them.
# The remaining cells are ready to run without student edits.

EXPLORATION = 0.10
EPISODES = 100
LEARNING_RATE = 0.00025

### check the choices

These checks catch typos before a long experiment starts. They do not change your settings. They just judge you silently, like a critic.

In [ ]:
import math
assert (
    isinstance(EXPLORATION, (int, float))
    and not isinstance(EXPLORATION, bool)
    and math.isfinite(EXPLORATION) and 0 <= EXPLORATION <= 1
), "Exploration must be between 0 and 1."
assert (
    isinstance(EPISODES, int) and not isinstance(EPISODES, bool) and EPISODES > 0
), "Episodes must be a positive whole number."
assert (
    isinstance(LEARNING_RATE, (int, float))
    and not isinstance(LEARNING_RATE, bool)
    and math.isfinite(LEARNING_RATE) and 0 < LEARNING_RATE <= 1
), "Learning rate must be greater than 0 and at most 1."
print(f"Your experiment (Taylor's Version): {EXPLORATION:.0%} exploration, {EPISODES} episodes, learning rate {LEARNING_RATE:g}.")

## 2. automatic setup

Run this once per fresh session. It installs missing packages into the notebook's Python environment.
Python 3.11–3.13 is recommended. No separate game download is needed, because the Atari ROM comes with `ale-py`.
For local popup playback, keep `pacman_player.py` next to this notebook and use a Python kernel with Tk. Colab and machines without a desktop just get the fast inline preview.
If a package was already imported before an upgrade, restart the kernel and choose Run All again. Turning it off and on again: still undefeated.

In [ ]:
%pip -q install "torch>=2.6,<3" "gymnasium==1.3.0" "ale-py==0.11.2" "opencv-python-headless==4.14.0.94" "numpy>=2,<3" "matplotlib>=3.9,<4" "Pillow>=10,<13"

### load the tools

Gymnasium runs the game, PyTorch does the learning, Pillow saves gameplay, and OpenCV helps find Ms. Pac-Man on screen so she can be replaced. This cell also prints the actual Python and package versions, so runs can be compared.

In [ ]:
import csv
import math
import json
import platform
import random
import shutil
import time
from collections import deque
from datetime import datetime
from pathlib import Path
from importlib.metadata import version

import ale_py
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
import cv2
from PIL import Image as PILImage
from IPython.display import Image, FileLink, display

gym.register_envs(ale_py)
# Keep CPU thread overhead manageable on laptops.
torch.set_num_threads(min(4, torch.get_num_threads()))


VERSIONS = {p: version(p) for p in ['torch', 'gymnasium', 'ale-py', 'opencv-python-headless', 'numpy', 'matplotlib', 'Pillow']}
print(f"Python {platform.python_version()} | {platform.platform()}")
print(VERSIONS)

### fixed experiment settings

These values keep experiments comparable. One **decision** moves the game forward four frames. The agent remembers the last 5,000 decisions and studies 32 of them at a time. The first 1,000 decisions only collect experience; after that, it learns every four decisions. You only need to edit the three choices in section 1. Touch these and the whole tour gets rescheduled.

In [ ]:
# Fixed classroom settings. Students only need the three choices above.
SEED = 42
MAX_STEPS = 3000          # Agent decisions per game, about 200 seconds of game time.
FRAME_SKIP = 4           # Four emulator frames per agent decision, applied once.
REPLAY_CAPACITY = 5000   # About 168 MiB of pixel data, allocated gradually.
BATCH_SIZE = 32
WARMUP_STEPS = 1000      # Random-action collection before any weight updates.
TRAIN_EVERY = 4          # One update every four training decisions.
TARGET_EVERY = 1000      # Copy learned weights to target every 1,000 decisions.
GAMMA = 0.99             # How much future rewards count.
DEMO_EVERY = 25          # Episodes between gameplay demonstrations.
EVAL_SEEDS = [101, 202, 303, 404, 505]
EVAL_EXPLORATION = 0.05  # Identical before/after; independent of training choice.


### faster sample playback (plus the psychedelic switch)

Samples play at **4× speed**: 20 seconds of game time takes about five seconds to watch. This only changes the recording and playback; the game itself runs normally.

Local samples open in an **always-on-top popup** with Pause, Replay, and a pin toggle. Each sample plays twice and stops. Set `SHOW_POPUPS = False` for inline playback only.

`SWIFT_MODE = True` turns on the Taylor's Version makeover for every GIF. Set it to `False` to get the original, deeply beige Atari footage, which is useful if a grader has a headache.

In [ ]:
PREVIEW_SPEED = 4
PREVIEW_PLAYS = 2
SHOW_POPUPS = True
SWIFT_MODE = True        # False = original beige Atari footage
PREVIEW_SECONDS = 20
PREVIEW_STRIDE = PREVIEW_SPEED
PREVIEW_DECISIONS = round(PREVIEW_SECONDS * 60 / FRAME_SKIP)
PREVIEW_FRAME_MS = round(1000 * FRAME_SKIP / 60)

## 3. the game and the learning rule (the actual AI part, no glitter)

The network sees **four grayscale 84 × 84 screens** so it can tell which way things are moving.
Its outputs are **Q-values**: estimated future points for each of the 9 joystick moves. They are guesses of value, not probabilities.

The agent keeps past experiences in a **replay memory** and studies random handfuls of them.
A second, slower-changing **target network** supplies the answer key:

**target = reward + discounted value of the next screen**

At game over there is no future reward, so that term is dropped. At the time limit the game *could* have continued, so the future-value term stays.
During learning, rewards are clipped to −1 through +1, so a 10-point Swiftie and a 1,600-point ghost combo feel about the same to the network. It is a humble network. Every score **you** see is the original game score.
The code below is ready to use. You do not need to edit it. You should read it, though; that's the assignment.

### 3a. turn game screens into observations

Start Ms. Pac-Man, shrink each screen to 84 × 84 grayscale, and stack four screens. One image shows where things are; four in a row also show which way they're moving. Reset starts with four copies of the first screen.
Note: this is the screen the AI learns from. No Taylor, no Swifties, no money. Just vibes, in gray.

In [ ]:
def make_env():
    env = gym.make("ALE/MsPacman-v5", frameskip=1,
                   repeat_action_probability=0.25, render_mode="rgb_array")
    env = gym.wrappers.AtariPreprocessing(
        env, noop_max=30, frame_skip=FRAME_SKIP, screen_size=84,
        terminal_on_life_loss=False, grayscale_obs=True, scale_obs=False)
    env = gym.wrappers.FrameStackObservation(env, stack_size=4, padding_type="reset")
    return gym.wrappers.TimeLimit(env, max_episode_steps=MAX_STEPS)

### 3b. estimate the value of each move

Convolution layers look for patterns (walls, dots, ghosts) in the screens. The final layer outputs one number per move: "how many points do I expect if I do this?" `forward` scales pixels from 0–255 to 0–1 first.

In [ ]:
class DQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        # Read visual patterns, then produce one value per available move.
        self.layers = nn.Sequential(
            nn.Conv2d(4, 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1), nn.ReLU(),
            nn.Flatten(), nn.Linear(3136, 512), nn.ReLU(),
            nn.Linear(512, n_actions))

    def forward(self, screens):
        return self.layers(screens.float() / 255.0)

### 3c. remember past experience

Each memory stores the screen stack, move, reward, next screen, and whether the game ended. Studying *random* old memories stops the network from obsessing over the last few seconds. Consecutive stacks share three frames, so only the one new frame is stored, to save memory.

In [ ]:
class ReplayMemory:
    def __init__(self, capacity):
        self.items = deque(maxlen=capacity)

    def add(self, obs, action, reward, next_obs, terminated, truncated):
        # The next stack is [obs[1], obs[2], obs[3], new_frame].
        # Store five frames instead of two overlapping four-frame stacks.
        self.items.append((np.array(obs, dtype=np.uint8, copy=True), action,
                           float(np.clip(reward, -1, 1)),
                           np.array(next_obs[-1], dtype=np.uint8, copy=True),
                           bool(terminated), bool(truncated)))

    def sample(self, size, rng):
        batch = rng.sample(list(self.items), size)
        obs, actions, rewards, last_frames, ended, truncated = zip(*batch)
        obs = np.stack(obs)
        next_obs = np.concatenate([obs[:, 1:], np.stack(last_frames)[:, None]], axis=1)
        return obs, np.array(actions), np.array(rewards, dtype=np.float32), next_obs, np.array(ended), np.array(truncated)

    def __len__(self):
        return len(self.items)

### 3d. choose a move (epsilon-greedy)

With probability `epsilon`, pick a random move (**explore**: try something new). Otherwise pick the move with the highest predicted value (**exploit**: play the hits). Choosing a move does not change the network.

In [ ]:
@torch.no_grad()
def choose_action(model, obs, epsilon, rng, n_actions):
    if rng.random() < epsilon:
        return rng.randrange(n_actions)
    device = next(model.parameters()).device
    screens = torch.as_tensor(np.asarray(obs), device=device).unsqueeze(0)
    return int(model(screens).argmax(dim=1).item())

### 3e. learn from a batch

1. Predict the value of each remembered move.
2. Build a target from its reward plus the next screen's estimated value (the **Bellman equation**, the one bit of 1950s math doing all the work here).
3. Measure the error with **Huber loss** and nudge the network with **Adam**.

The target network changes less often, so learning isn't chasing its own tail. Real game over removes future reward; hitting the time limit does not.

In [ ]:
def learn(model, target, optimizer, batch):
    device = next(model.parameters()).device
    obs, actions, rewards, next_obs, ended, truncated = [torch.as_tensor(x, device=device) for x in batch]
    # 1. What value did the network predict for the move we took?
    values = model(obs).gather(1, actions.long().unsqueeze(1)).squeeze(1)
    # 2. What target does the observed reward suggest?
    with torch.no_grad():
        # Mask true termination only. A time limit still allows bootstrapping.
        targets = rewards + GAMMA * (~ended).float() * target(next_obs).max(dim=1).values
    # 3. Reduce the gap between prediction and target.
    loss = nn.functional.smooth_l1_loss(values, targets)
    if not torch.isfinite(loss):
        raise RuntimeError("Loss became non-finite. Start a new run with a smaller learning rate.")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 10.0, error_if_nonfinite=True)
    optimizer.step()
    return float(loss.item())

### 3f. select working hardware

Try CUDA (NVIDIA), Apple MPS, then CPU. A real training batch is tested on each before committing. Trust, but verify.

In [ ]:
def select_device(n_actions):
    candidates = []
    if torch.cuda.is_available():
        candidates.append("cuda")
    if torch.backends.mps.is_available():
        candidates.append("mps")
    candidates.append("cpu")
    for candidate in candidates:
        try:
            # Test the actual network, minibatch, backward pass, and optimizer.
            probe = DQN(n_actions).to(candidate)
            optimizer = torch.optim.Adam(probe.parameters(), lr=LEARNING_RATE)
            screens = np.zeros((BATCH_SIZE, 4, 84, 84), dtype=np.uint8)
            batch = (screens, np.zeros(BATCH_SIZE, dtype=np.int64),
                     np.ones(BATCH_SIZE, dtype=np.float32), screens,
                     np.zeros(BATCH_SIZE, dtype=bool), np.zeros(BATCH_SIZE, dtype=bool))
            learn(probe, probe, optimizer, batch)
            print(f"Using {candidate.upper()}; training-batch check passed.")
            return torch.device(candidate)
        except RuntimeError as error:
            if candidate == "cpu":
                raise
            print(f"{candidate.upper()} check failed ({error}). Trying the next device.")

### check the first screen

Create the game, check the four-screen stack, and list the joystick moves. If this cell succeeds, the game and the hardware are ready for their close-up.

In [ ]:
probe_env = make_env()
try:
    probe_obs, _ = probe_env.reset(seed=SEED)
    assert probe_obs.shape == (4, 84, 84) and probe_obs.dtype == np.uint8
    assert all(np.array_equal(probe_obs[0], frame) for frame in probe_obs)
    N_ACTIONS = int(probe_env.action_space.n)
    DEVICE = select_device(N_ACTIONS)
    print("Available moves:", probe_env.unwrapped.get_action_meanings())
    display(PILImage.fromarray(probe_env.render()).resize((320, 420)))
finally:
    probe_env.close()
print(f"Replay pixel budget: {REPLAY_CAPACITY * 5 * 84 * 84 / 2**20:.0f} MiB, plus model and batch memory.")

## 3½. SWIFT-MAN (Taylor's Version): the makeover

This section is **purely cosmetic**. It repaints recorded GIF frames and nothing else. The neural network never calls it and never sees its output.
It reads the full-color Atari frame, finds things by color, and paints over them:

- **Ms. Pac-Man → 8-bit Taylor Swift.** A Mario/Luigi-style side-view sprite with bangs, cat-eye liner, red lip, a mic, a sequined bodysuit, and a two-frame walk cycle (she also chomps, it's still Pac-Man). She faces the way she's moving and leaves a rainbow afterimage trail.
- **Pellets → tiny Swifties and bags of money**, in a checkerboard. When she passes one, it pops a `+1 SWIFTIE` or `+$1M` and the counters go up.
- **Power pellets → giant money bags** with spinning rainbow rays. Eating one triggers `BAG SECURED. MARKETS REACT.` and a strobe.
- **Ghosts → HATERS, PAPARAZZI, TICKET BOTS, CRITICS**, wearing sunglasses, carrying cameras, and firing flash photography.
- **The maze → a rainbow plasma**, with chromatic aberration, torn scanlines, random color inversion, screen shake when a ghost gets eaten, and a CRT filter.
- **The score bar → a HUD** with a (fake, vibes-based) net worth counter, RL captions, and a news ticker.

The sprites are text maps, one letter per pixel, exactly how 1980s sprites were designed. Edit a letter in `TAYLOR_TOP` and you've done a re-recording.

In [ ]:
# Taylor's Version, part 1: the cast, in genuine 8-bit pixel art.
# Each sprite is a little text map, one letter per pixel, like a 1985 plumber but with better bangs.
# Parody. Not affiliated with, endorsed by, or known to Taylor Swift or her lawyers.
from PIL import ImageDraw, ImageFont, ImageOps

SKIN_SCALE = 2            # GIFs are 320 × 420: twice the Atari screen, same as the popup.
PLAY_ROWS = 172           # Rows 0–171 are the maze; below that is the Atari score bar.
PAC_RGB = (210, 164, 74)  # Ms. Pac-Man's yellow. She has been replaced. She was not consulted.
GHOST_NAMES = {           # Atari ghost colors → who they really are.
    (200, 72, 72): "HATERS", (198, 89, 179): "PAPARAZZI",
    (84, 184, 153): "TICKET BOTS", (180, 122, 48): "CRITICS",
}
PALETTE = {  # One NES-ish palette for the whole cast. "." is transparent.
    "H": (248, 208, 96), "h": (200, 144, 40),     # blonde hair, hair shadow
    "S": (252, 212, 164), "K": (24, 16, 32),       # skin, outline / eyeliner
    "E": (40, 96, 232), "R": (232, 0, 56),         # blue eyes, THE red lip
    "P": (168, 40, 224), "p": (255, 150, 255),     # sequined bodysuit, sequins
    "B": (240, 24, 104), "W": (255, 255, 255),     # sparkly boots, sparkle
    "M": (96, 96, 112), "Y": (255, 214, 0),        # microphone, gold
    "t": (204, 150, 72), "D": (0, 160, 64),        # money-bag burlap, dollar green
    "L": (110, 200, 255),                          # camera lens
    "1": (255, 105, 180), "2": (150, 90, 255), "3": (0, 200, 200),  # Swiftie merch colors
    "4": (120, 70, 30), "5": (30, 20, 20), "6": (141, 85, 36),       # Swiftie hair / skin
}


def skin_font(size):
    try:
        return ImageFont.load_default(size=size)
    except TypeError:  # Pillow < 10.1 has one tiny font and no opinions.
        return ImageFont.load_default()


def pixel_sprite(rows, px=SKIN_SCALE, swap=None):
    """Turn a text map into a crisp RGBA sprite, every art pixel drawn as a px × px block."""
    swap = swap or {}
    art = np.zeros((len(rows), len(rows[0]), 4), np.uint8)
    for y, row in enumerate(rows):
        for x, key in enumerate(row):
            if key != ".":
                art[y, x] = PALETTE[swap.get(key, key)] + (255,)
    return PILImage.fromarray(art.repeat(px, 0).repeat(px, 1), "RGBA")


# Side view, facing right, 14 × 18 art pixels: bangs, cat-eye liner, red lip, mic, sequins, boots.
TAYLOR_TOP = [
    "....KKKKKK....",
    "...KHHHHHHKK..",
    "..KHHHHHHHHHK.",
    "..KHHHHHHHHHHK",
    "..KHHhHHHHHHK.",
    ".KHHhSSKKSSSK.",
    ".KHHhSSSESSSSK",
    ".KHHhhSSSSSSK.",
]
TAYLOR_MOUTH = {False: ".KHHhhSSSRRK..", True: ".KHHhhSSSRKR.."}  # chomp, Pac-Man-style
TAYLOR_BODY = [
    ".KHHHhhSSSK.MM",
    ".KHHHhKSSK.SM.",
    "..KHKPPPPPPSK.",
    "..KKPpPPPpPK..",
    "...KPPPpPPPK..",
    "...KPpPPPpPK..",
]
TAYLOR_LEGS = {  # two-frame walk cycle
    0: ["....KSSKSSK...", "....KBBKBBK...", "...KBBBKBBBK.."],
    1: ["...KSSK.KSSK..", "..KBBK...KBBK.", ".KBBBK...KBBBK"],
}
SWIFTIE = [  # a tiny fan, arm up, friendship bracelet (Y) on the wrist
    "..hhh.S",
    ".hSSSh Y".replace(" ", ""),
    ".hSSSh.",
    "..SSS.S",
    ".TTTTTT",
    ".TTRTT.",
    "..TTTT.",
    "..S..S.",
]
MONEY_MAP = [
    "..KKK..",
    "..tYt..",
    ".KtDtK.",
    "KtDDDtK",
    "KtDtttK",
    "KttDttK",
    "KtttDtK",
    "KtDDDtK",
    ".KKKKK.",
]
SHADES_MAP = ["KKKKKKKK", "KWK..KWK", ".K....K."]
CAMERA_MAP = [".KK...", "KKKKKK", "KKLLKK", "KKLLKK", "KKKKKK"]


def _taylor(walk, mouth):
    """Taylor plus a glowing halo, so she stays visible inside the fever dream. Main-character lighting."""
    body = pixel_sprite(TAYLOR_TOP + [TAYLOR_MOUTH[mouth]] + TAYLOR_BODY + TAYLOR_LEGS[walk])
    alpha = np.pad(np.asarray(body.getchannel("A")), 3)
    glow = cv2.dilate(alpha, np.ones((5, 5), np.uint8))
    halo = np.zeros(glow.shape + (4,), np.uint8)
    halo[glow > 0] = (255, 255, 200, 220)
    framed = PILImage.fromarray(halo, "RGBA")
    framed.alpha_composite(body, (3, 3))
    return framed


def _tinted(sprite, rgb, alpha):
    """Recolor a sprite and fade it: one ghostly afterimage of Taylor for the trail."""
    solid = PILImage.new("RGBA", sprite.size, rgb + (0,))
    solid.putalpha(sprite.getchannel("A").point(lambda a: int(a * alpha)))
    return solid


# Frame f alternates walk pose and chomp together, so she walks AND eats. Multitasking queen.
TAYLOR = {(facing, f): (lambda im: ImageOps.mirror(im) if facing < 0 else im)(_taylor(f, bool(f)))
          for facing in (1, -1) for f in (0, 1)}
TRAIL_COLORS = [(255, 0, 200), (0, 240, 255), (255, 240, 0), (120, 255, 60), (255, 90, 0), (150, 80, 255)]
TRAILS = {facing: [_tinted(TAYLOR[(facing, 0)], c, 0.55 - 0.08 * i) for i, c in enumerate(TRAIL_COLORS)]
          for facing in (1, -1)}
SWIFTIES = [pixel_sprite(SWIFTIE, swap={"h": hair, "S": skin, "T": shirt})
            for hair, skin, shirt in [("4", "S", "1"), ("5", "6", "2"), ("H", "S", "3"),
                                      ("1", "S", "Y"), ("5", "S", "D"), ("4", "6", "R")]]
MONEY_BAG, BIG_MONEY_BAG = pixel_sprite(MONEY_MAP), pixel_sprite(MONEY_MAP, px=3)
SHADES, CAMERA = pixel_sprite(SHADES_MAP), pixel_sprite(CAMERA_MAP)
print("Cast assembled:", len(SWIFTIES), "Swifties, 2 sizes of money bag, 1 8-bit Taylor,",
      "4 ghosts (now haters, paparazzi, ticket bots, and critics).")

### the stage: turning an Atari frame into a fever dream

`SwiftSkin.render(frame, score)` does the repainting. It uses color matching and OpenCV "connected components" (blobs of same-colored pixels) to find walls, pellets (small blobs), power pellets (slightly taller blobs), Ms. Pac-Man (her yellow), and ghosts (8 × 10 blobs in ghost colors).
Then it paints plasma, sprites, labels, and glitches. It runs about 40 ms per frame, and only on the ~75 frames per recorded clip.

In [ ]:
# Taylor's Version, part 2: the stage. Called once per recorded GIF frame, never during learning.
CAPTIONS = [
    "Q-VALUES (TAYLOR'S VERSION)",
    "WELCOME TO THE EPSILON-GREEDY ERA",
    "RANDOM MOVES ARE NOW 'ARTISTIC CHOICES'",
    "GAMMA = 0.99: SHE PLANS 12 ALBUMS AHEAD",
    "REPLAY BUFFER NEVER FORGETS. NOR DO FANS.",
    "TARGET NETWORK SYNCED. EASTER EGG FOUND.",
    "REWARDS CLIPPED TO [-1, 1]. PUBLICIST SOBS.",
    "THE NETWORK SEES GRAYSCALE. LUCKY NETWORK.",
    "LOSS WENT DOWN. SCORE? NO COMMENT.",
    "HUBER LOSS: FOR MISTAKES THAT GET LOUD",
    "GHOSTS HAVE BEEN SERVED. LEGALLY.",
    "4 FRAMES STACKED. 13 WOULD BE LUCKIER.",
    "BELLMAN WOULD BE PROUD. OR CONCERNED.",
    "NO SWIFTIES HARMED. THEY WERE RECRUITED.",
    "ATARI 2600, BUT MAKE IT A STADIUM TOUR",
    "RE-RECORDING THIS MAZE FROM MEMORY",
    "NOT FINANCIAL ADVICE. BARELY A GAME.",
]
TICKER = "   ///   ".join(CAPTIONS) + "   ///   "


class SwiftSkin:
    """Repaints one Ms. Pac-Man game as a psychedelic 8-bit Taylor Swift fever dream.

    Make a new SwiftSkin for every recorded game: it remembers Taylor's trail,
    who got recruited, and how many bags were secured.
    """

    def __init__(self, seed=0):
        h, w = PLAY_ROWS * SKIN_SCALE, 160 * SKIN_SCALE
        self.yy, self.xx = np.mgrid[0:h, 0:w].astype(np.float32)
        self.radius = np.hypot(self.xx - w / 2, self.yy - h / 2)
        self.rng = random.Random(seed)
        self.t, self.last_score = 0, 0.0
        self.trail, self.facing, self.last_pac = deque(maxlen=len(TRAILS[1])), 1, None
        self.last_pellets, self.swifties, self.bags = set(), 0, 0
        self.floaters, self.shout, self.shake, self.strobe = [], None, 0, 0
        self.font, self.small, self.big = skin_font(11), skin_font(9), skin_font(22)

    # ---- reading the Atari screen -------------------------------------------------
    def _find(self, play):
        """Locate maze walls, pellets, power pellets, Ms. Pac-Man, and ghosts by color."""
        colors, counts = np.unique(play[::2, ::2].reshape(-1, 3), axis=0, return_counts=True)
        order = [tuple(int(v) for v in colors[i]) for i in np.argsort(-counts)]
        order = [c for c in order if c != (0, 0, 0)]
        background, wall = order[0], order[1]
        wall_mask = np.all(play == wall, axis=2).astype(np.uint8)
        n, labels, stats, centers = cv2.connectedComponentsWithStats(wall_mask, connectivity=8)
        pellets, powers, dots = [], [], []
        for i in range(1, n):
            x, y, w, h, area = stats[i]
            if area <= 10 and w <= 4 and h <= 3:
                pellets.append((int(centers[i][0]), int(centers[i][1]))); dots.append(i)
            elif w <= 5 and 5 <= h <= 8 and area <= 36:
                powers.append((int(centers[i][0]), int(centers[i][1]))); dots.append(i)
        maze = wall_mask.astype(bool) & ~np.isin(labels, dots)
        pac = np.all(play == PAC_RGB, axis=2)
        ghosts = []
        other = ~np.all(play == background, axis=2) & ~wall_mask.astype(bool) & ~pac & play.any(axis=2)
        n, labels, stats, centers = cv2.connectedComponentsWithStats(other.astype(np.uint8), connectivity=8)
        for i in range(1, n):
            x, y, w, h, area = stats[i]
            if area >= 25 and 6 <= w <= 9 and 7 <= h <= 11:
                color = tuple(int(v) for v in play[labels == i][0])
                name = GHOST_NAMES.get(color, "SHOOK" if color[2] > max(color[:2]) else None)
                if name:
                    ghosts.append((x, y, w, h, name))
        pac_box = None
        if pac.any():
            ys, xs = np.nonzero(pac)
            pac_box = (int(xs.mean()), int(ys.mean()))
        return maze, pellets, powers, pac_box, ghosts, other

    # ---- painting -------------------------------------------------------------------
    def _plasma(self, maze):
        """The psychedelic part: a drifting rainbow plasma under neon walls."""
        t = self.t * (5 if self.strobe else 1)
        wave = (np.sin(self.xx * 0.045 + t * 0.35) + np.sin(self.yy * 0.06 - t * 0.27)
                + np.sin((self.xx + self.yy) * 0.03 + t * 0.2) + np.sin(self.radius * 0.05 - t * 0.4))
        hue = ((wave + 4) * 22.5 + t * 6) % 180
        big_maze = maze.repeat(SKIN_SCALE, 0).repeat(SKIN_SCALE, 1)
        hsv = np.empty(hue.shape + (3,), np.uint8)
        hsv[..., 0] = np.where(big_maze, (hue + 90) % 180, hue)
        hsv[..., 1] = np.where(big_maze, 170, 255)
        hsv[..., 2] = np.where(big_maze, 255, 70)
        rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
        outline = cv2.dilate(big_maze.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool) & ~big_maze
        rgb[outline] = (10, 0, 25)
        return rgb

    def _stamp(self, canvas, sprite, cx, cy):
        canvas.alpha_composite(sprite, (int(cx - sprite.width / 2), int(cy - sprite.height / 2)))

    def _hud(self, canvas, score):
        d = ImageDraw.Draw(canvas)
        top = PLAY_ROWS * SKIN_SCALE
        d.rectangle((0, top, canvas.width, canvas.height), fill=(12, 0, 24))
        worth = 1_600_000_000 + int(score) * 1_000_000
        d.text((6, top + 3), f"SCORE {int(score):,}", fill=(255, 230, 90), font=self.font)
        d.text((canvas.width - 6, top + 3), f"NET WORTH ${worth:,}", fill=(90, 255, 140),
               font=self.font, anchor="ra")
        d.text((6, top + 19), f"SWIFTIES RECRUITED {self.swifties}   BAGS SECURED {self.bags}",
               fill=(255, 140, 230), font=self.font)
        caption = CAPTIONS[(self.t // 12 + self.rng_offset) % len(CAPTIONS)]
        d.text((canvas.width / 2, top + 42), caption, fill=self._rainbow(0), font=self.font, anchor="ma")
        shift = (self.t * 9) % (len(TICKER) * 5)
        d.text((6 - shift, top + 60), TICKER * 2, fill=(150, 150, 190), font=self.small)

    def _rainbow(self, offset):
        hsv = np.uint8([[[(self.t * 13 + offset) % 180, 200, 255]]])
        return tuple(int(v) for v in cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)[0, 0])

    def _glitch(self, frame):
        """The unhinged part: chromatic aberration, torn scanlines, strobes, and shaking."""
        s = 1 + 3 * (self.t % 7 == 0) + 2 * bool(self.strobe)
        frame[..., 0] = np.roll(frame[..., 0], s, axis=1)
        frame[..., 2] = np.roll(frame[..., 2], -s, axis=1)
        if self.t % 9 == 4 or self.shake:
            for _ in range(3):
                y = self.rng.randrange(0, frame.shape[0] - 24)
                h = self.rng.randrange(4, 24)
                frame[y:y + h] = np.roll(frame[y:y + h], self.rng.randrange(-20, 21), axis=1)
        if self.t % 29 == 13:
            frame[:PLAY_ROWS * SKIN_SCALE] = 255 - frame[:PLAY_ROWS * SKIN_SCALE]
        if self.shake:
            frame = np.roll(frame, (self.rng.randrange(-5, 6), self.rng.randrange(-5, 6)), axis=(0, 1))
            self.shake -= 1
        frame[1::2] = (frame[1::2] * 0.8).astype(np.uint8)  # CRT scanlines, for authenticity
        return frame

    def render(self, rgb, score):
        """Atari RGB frame (210 × 160) + score so far → one Taylor's Version GIF frame."""
        if self.t == 0:
            self.rng_offset = self.rng.randrange(len(CAPTIONS))
        maze, pellets, powers, pac, ghosts, other = self._find(rgb[:PLAY_ROWS])
        gained = score - self.last_score
        if gained >= 200:
            self.shout, self.shake = ("PAPARAZZO DEFEATED", 8), 6
        elif gained >= 50 and not self.strobe and len(powers) < getattr(self, "last_powers", 4):
            self.shout, self.strobe = ("BAG SECURED. MARKETS REACT.", 8), 8
        self.last_powers = len(powers)

        stage = np.zeros((210 * SKIN_SCALE, 160 * SKIN_SCALE, 3), np.uint8)
        stage[:PLAY_ROWS * SKIN_SCALE] = self._plasma(maze)
        keep = other.repeat(SKIN_SCALE, 0).repeat(SKIN_SCALE, 1)  # ghosts and fruit keep their pixels
        stage[:PLAY_ROWS * SKIN_SCALE][keep] = rgb[:PLAY_ROWS].repeat(SKIN_SCALE, 0).repeat(SKIN_SCALE, 1)[keep]
        canvas = PILImage.fromarray(stage).convert("RGBA")

        current = set(pellets)
        for x, y in pellets:  # checkerboard: half Swifties, half money
            kind = (x // 8 + y // 12) % 2
            sprite = MONEY_BAG if kind else SWIFTIES[(x * 7 + y * 3) % len(SWIFTIES)]
            self._stamp(canvas, sprite, x * SKIN_SCALE, y * SKIN_SCALE + (self.t + x) % 2)
        if pac and self.last_pellets:
            for x, y in self.last_pellets - current:
                if abs(x - pac[0]) + abs(y - pac[1]) <= 24:
                    money = (x // 8 + y // 12) % 2
                    self.bags += money
                    self.swifties += 1 - money
                    self.floaters.append(["+$1M" if money else "+1 SWIFTIE",
                                          pac[0] * SKIN_SCALE, pac[1] * SKIN_SCALE - 14, 5])
        self.last_pellets = current if pac else self.last_pellets

        d = ImageDraw.Draw(canvas)
        for x, y in powers:
            cx, cy = x * SKIN_SCALE, y * SKIN_SCALE
            spin = self.t * 0.5
            for k in range(8):
                a = spin + k * math.pi / 4
                d.line((cx, cy, cx + 20 * math.cos(a), cy + 20 * math.sin(a)), fill=self._rainbow(k * 20), width=2)
            self._stamp(canvas, BIG_MONEY_BAG, cx, cy)

        for x, y, w, h, name in ghosts:
            cx, top = (x + w / 2) * SKIN_SCALE, y * SKIN_SCALE
            self._stamp(canvas, SHADES, cx, top + 7)
            self._stamp(canvas, CAMERA, cx + 12, top + 14)
            if (self.t + x) % 5 == 0:  # flash photography, always
                d.ellipse((cx + 4, top - 2, cx + 26, top + 20), fill=(255, 255, 255, 170))
            d.text((cx, top - 3), name, fill=(255, 255, 255), font=self.small, anchor="md",
                   stroke_width=2, stroke_fill=(0, 0, 0))

        if pac:
            if self.last_pac and pac[0] != self.last_pac[0]:
                self.facing = 1 if pac[0] > self.last_pac[0] else -1
            for i, (tx, ty, facing) in enumerate(reversed(self.trail)):
                self._stamp(canvas, TRAILS[facing][i], tx, ty)
            cx, cy = pac[0] * SKIN_SCALE, pac[1] * SKIN_SCALE - 4
            self._stamp(canvas, TAYLOR[(self.facing, self.t % 2)], cx, cy)
            self.trail.append((cx, cy, self.facing))
            self.last_pac = pac

        for f in self.floaters:
            d.text((f[1], f[2]), f[0], fill=self._rainbow(f[2]), font=self.small, anchor="ms",
                   stroke_width=2, stroke_fill=(0, 0, 0))
            f[2] -= 4; f[3] -= 1
        self.floaters = [f for f in self.floaters if f[3] > 0]

        if self.t < 8:
            d.text((canvas.width / 2, 120), "SWIFT-MAN", fill=self._rainbow(0), font=self.big,
                   anchor="mm", stroke_width=3, stroke_fill=(0, 0, 0))
            d.text((canvas.width / 2, 146), "(TAYLOR'S VERSION)", fill=self._rainbow(60), font=self.font,
                   anchor="mm", stroke_width=2, stroke_fill=(0, 0, 0))
        if self.shout:
            text, ttl = self.shout
            d.text((canvas.width / 2, 170), text, fill=self._rainbow(90), font=self.font,
                   anchor="mm", stroke_width=3, stroke_fill=(0, 0, 0))
            self.shout = (text, ttl - 1) if ttl > 1 else None

        self._hud(canvas, score)
        frame = self._glitch(np.asarray(canvas.convert("RGB")).copy())
        self.strobe = max(0, self.strobe - 1)
        self.last_score = score
        self.t += 1
        return PILImage.fromarray(frame)

### preview the makeover

One frame, side by side with the original. If this looks deranged, it's working.

In [ ]:
# A quick look before the tour: a few random moves, then the same frame, before and after.
preview_env = gym.make("ALE/MsPacman-v5", frameskip=4, repeat_action_probability=0.0, render_mode="rgb_array")
preview_env.reset(seed=SEED)
preview_skin = SwiftSkin(SEED)
for _ in range(6):
    preview_env.step(preview_env.action_space.sample())
    atari_frame = preview_env.render()
    preview_frame = preview_skin.render(atari_frame, 0.0)
preview_env.close()
side_by_side = PILImage.new("RGB", (160 * SKIN_SCALE * 2 + 10, 210 * SKIN_SCALE), (0, 0, 0))
side_by_side.paste(PILImage.fromarray(atari_frame).resize(
    (160 * SKIN_SCALE, 210 * SKIN_SCALE), PILImage.Resampling.NEAREST), (0, 0))
side_by_side.paste(preview_frame, (160 * SKIN_SCALE + 10, 0))
print("Left: what Atari shows. Right: what you'll see. (The network sees neither: it sees 84 × 84 gray.)")
display(side_by_side)

## 4. gameplay, scores, and saved results

Evaluation uses a separate game and the same five seeds before and after training.
Both use 5% random moves. Evaluation never changes the network or the training memory.
The baseline is an **untrained network**, not a random-action agent.

GIFs show the first 20 seconds at most, squeezed into about five seconds at 4× speed. The reported score covers the whole game.
Every GIF gets the SWIFT-MAN makeover while it's being recorded (see section 3½). Scores are computed from the untouched game.

### 4a. score complete games and record a short sample

Games run at full speed. We record a short clip but count points for the **whole game**. For the final preview we keep the clip from the highest-scoring game.
When `SWIFT_MODE` is on, each recorded frame goes through `SwiftSkin.render(...)` *after* the move is chosen from the plain screen. The AI never sees the makeover.

In [ ]:
def evaluate(model, seeds, gif_path=None, record_best=False):
    env = make_env()
    scores, lengths, capped = [], [], []
    best_score, chosen_frames = -float("inf"), []
    was_training = model.training
    model.eval()
    try:
        for index, seed in enumerate(seeds):
            rng = random.Random(seed + 10000)
            obs, _ = env.reset(seed=seed)
            total, frames = 0.0, []
            capture = gif_path is not None and (record_best or index == 0)
            skin = SwiftSkin(seed) if capture and SWIFT_MODE else None  # fresh stage per game
            for step in range(MAX_STEPS):
                if capture and step < PREVIEW_DECISIONS and step % PREVIEW_STRIDE == 0:
                    screen = env.render()  # full-color Atari frame, for humans only
                    frames.append(skin.render(screen, total) if skin else PILImage.fromarray(screen).copy())
                action = choose_action(model, obs, EVAL_EXPLORATION, rng, N_ACTIONS)
                obs, reward, ended, truncated, _ = env.step(action)
                total += reward
                if ended or truncated:
                    break
            scores.append(float(total))
            lengths.append(step + 1)
            capped.append(bool(truncated))
            if capture and (not record_best or total > best_score):
                chosen_frames, best_score = frames, total
    finally:
        env.close()
        model.train(was_training)
    if gif_path is not None and chosen_frames:
        chosen_frames[0].save(gif_path, save_all=True, append_images=chosen_frames[1:],
                              duration=PREVIEW_FRAME_MS, loop=PREVIEW_PLAYS - 1)
    return {"seeds": list(seeds), "scores": scores, "mean": float(np.mean(scores)),
            "steps": lengths, "time_limited": capped}

### 4b. show the sample

Always keep a playable GIF in the notebook. The inline GIF and the popup both play twice. Locally, the floating player runs as its own process, so the concert doesn't block the training.

In [ ]:
def show_sample(gif_path, label):
    """Display a recorded excerpt inline and, locally, above the notebook."""
    print(f"{'SWIFT-MAN · ' if SWIFT_MODE else ''}{label} | {PREVIEW_SPEED}× playback | {PREVIEW_PLAYS} plays | first {PREVIEW_SECONDS}s of game time")
    display(Image(filename=str(gif_path)))
    if not SHOW_POPUPS:
        return
    try:
        from pacman_player import show_popup
        opened = show_popup(gif_path, f"{label} · {PREVIEW_SPEED}× speed")
        if not opened:
            print("Popup unavailable in this kernel; use the fast inline preview.")
    except (ImportError, OSError) as error:
        print(f"Using the inline preview. Local popup helper unavailable: {error}")

### 4c. save a model for playback

A checkpoint stores the learned weights and the number of moves. Reloading it lets us replay the saved agent. It is not a save-state for resuming training.

In [ ]:
def save_checkpoint(model, path, episode, steps):
    # Portable playback checkpoint. This is not an exact training-resume snapshot:
    # replay, optimizer, and emulator state are intentionally not persisted.
    payload = {"model": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
               "n_actions": N_ACTIONS, "episode": episode, "steps": steps,
               "exploration": EXPLORATION, "learning_rate": LEARNING_RATE}
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)

### 4d. save one row per game

The CSV keeps each finished game's raw score, decisions, exploration, loss, and time. A game with no learning updates has no meaningful loss yet (it's still in its opening-act era).

In [ ]:
def save_metrics(rows, path):
    fields = ["episode", "score", "steps", "total_steps", "exploration", "mean_loss", "elapsed_seconds", "terminated", "truncated"]
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

### 4e. draw the learning dashboard

Three panels: scores, prediction error (loss), and exploration. Watch the **scores**, not just the loss. Lower loss means the network predicts its own future better, and that future may still be bad.

In [ ]:
def plot_training(rows, path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    if rows:
        episodes = [r["episode"] for r in rows]
        scores = [r["score"] for r in rows]
        smooth = [np.mean(scores[max(0, i - 24):i + 1]) for i in range(len(scores))]
        axes[0].plot(episodes, scores, alpha=0.35, label="Each training game")
        axes[0].plot(episodes, smooth, label="Up to 25-game average")
        axes[0].legend(fontsize=8)
        axes[1].plot(episodes, [r["mean_loss"] for r in rows])
        axes[2].plot(episodes, [r["exploration"] for r in rows])
    for ax, title in zip(axes, ["Raw training score", "Mean update loss", "Training exploration"]):
        ax.set(title=title, xlabel="Completed episode")
        ax.grid(alpha=0.2)
    axes[2].set_ylim(0, 1.05)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

## 5. train your agent (the tour)

The setup cell below starts a **fresh experiment** every time, using your three choices.
It records the untrained baseline, trains, and prints a line after every game plus an update every 500 decisions.
Every 25 games it shows a SWIFT-MAN progress clip and saves a checkpoint and plot.

To stop early, hit **Interrupt / Stop** once. The model and finished games are saved.
Then run the remaining cells to evaluate and download. A run that ends before warm-up finishes has no learning yet.
Each run gets its own folder, so older experiments stay around like vault tracks.

### 5a. start a fresh experiment

Create both networks, an empty replay memory, random seeds, and a new results folder. This cell resets training. To try again, rerun from here or Run All.

In [ ]:
# Each execution starts fresh, including network weights and replay memory.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
train_rng = random.Random(SEED)
replay_rng = random.Random(SEED + 1)
RUN_DIR = Path("pacman_runs") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "demos").mkdir()
model = DQN(N_ACTIONS).to(DEVICE)
target = DQN(N_ACTIONS).to(DEVICE)
target.load_state_dict(model.state_dict())
target.eval()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
replay = ReplayMemory(REPLAY_CAPACITY)
history, demo_history = [], []
total_steps, completed_episodes, updates = 0, 0, 0

_training_started = False

### 5b. record the settings and test saving

Save the hardware and settings next to the starting weights. Reload the weights right away to catch checkpoint problems before a long run.

In [ ]:
config = {"exploration": EXPLORATION, "episodes_requested": EPISODES, "learning_rate": LEARNING_RATE,
          "seed": SEED, "device": str(DEVICE), "python": platform.python_version(),
          "platform": platform.platform(), "packages": VERSIONS,
          "environment": "ALE/MsPacman-v5", "frame_skip": FRAME_SKIP,
          "sticky_action_probability": 0.25, "noop_max": 30, "grayscale_size": 84,
          "stack_size": 4, "terminal_on_life_loss": False, "max_decisions_per_game": MAX_STEPS,
          "replay_capacity": REPLAY_CAPACITY, "warmup_decisions": WARMUP_STEPS,
          "batch_size": BATCH_SIZE, "train_every_decisions": TRAIN_EVERY,
          "target_sync_decisions": TARGET_EVERY, "gamma": GAMMA,
          "training_reward_clipping": [-1, 1], "eval_exploration": EVAL_EXPLORATION,
          "eval_seeds": EVAL_SEEDS, "preview_seconds": PREVIEW_SECONDS,
          "preview_speed": PREVIEW_SPEED, "preview_plays": PREVIEW_PLAYS,
          "preview_stride": PREVIEW_STRIDE,
          "preview_frame_ms": PREVIEW_FRAME_MS}
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))
save_checkpoint(model, RUN_DIR / "untrained.pt", 0, 0)
# Check saving and loading before starting the long run.
roundtrip = torch.load(RUN_DIR / "untrained.pt", map_location="cpu", weights_only=True)
assert all(torch.equal(v.cpu(), roundtrip["model"][k]) for k, v in model.state_dict().items())
del roundtrip

### 5c. measure the starting point

Play five games with the untrained network. That's the "before" photo. The clip shows the first game: an 8-bit pop star with no idea what she's doing.

In [ ]:
print("Evaluating the untrained network on five games. She has never seen a maze before. This can take a few minutes.", flush=True)
baseline = evaluate(model, EVAL_SEEDS, RUN_DIR / "demos" / "episode_0000.gif")
(RUN_DIR / "baseline.json").write_text(json.dumps(baseline, indent=2))
print("Before training:", baseline["scores"], "| mean:", baseline["mean"])
show_sample(RUN_DIR / "demos" / "episode_0000.gif", "Before training (debut era)")

### 5d. learn during one game

Repeat **choose → play → remember → learn** until game over or the time limit. The counters run across all games, so warm-up happens once. Every 1,000 decisions, the learning network is copied into the target network.

In [ ]:
def train_one_episode(train_env, episode):
    """Collect experience and update the agent during one complete game."""
    global total_steps, updates
    obs, _ = train_env.reset(seed=SEED + episode)
    score, losses = 0.0, []
    for step in range(MAX_STEPS):
        epsilon = 1.0 if total_steps < WARMUP_STEPS else EXPLORATION
        action = choose_action(model, obs, epsilon, train_rng, N_ACTIONS)
        next_obs, reward, ended, truncated, _ = train_env.step(action)
        replay.add(obs, action, reward, next_obs, ended, truncated)
        obs = next_obs
        score += reward
        total_steps += 1
        if total_steps >= WARMUP_STEPS and len(replay) >= BATCH_SIZE and total_steps % TRAIN_EVERY == 0:
            losses.append(learn(model, target, optimizer, replay.sample(BATCH_SIZE, replay_rng)))
            updates += 1
        if total_steps % TARGET_EVERY == 0:
            target.load_state_dict(model.state_dict())
        if total_steps % 500 == 0:
            elapsed = time.monotonic() - started
            print(f"  Episode {episode}/{EPISODES} | {total_steps:,} decisions | {updates:,} updates | {elapsed / 60:.1f} min", flush=True)
        if ended or truncated:
            break
    return {"episode": episode, "score": float(score), "steps": step + 1,
            "total_steps": total_steps, "exploration": epsilon,
            "mean_loss": float(np.mean(losses)) if losses else float("nan"),
            "elapsed_seconds": time.monotonic() - started,
            "terminated": bool(ended), "truncated": bool(truncated)}

### 5e. save a progress sample

Every 25 games, play one separate game, show the clip, and save a checkpoint. This doesn't add memories or change weights.

In [ ]:
def save_progress_sample(episode):
    gif_path = RUN_DIR / "demos" / f"episode_{episode:04d}.gif"
    demo = evaluate(model, EVAL_SEEDS[:1], gif_path)
    demo_history.append({"episode": episode, **demo})
    (RUN_DIR / "demo_scores.json").write_text(json.dumps(demo_history, indent=2))
    print("Separate demonstration score:", demo["scores"][0])
    save_checkpoint(model, RUN_DIR / f"episode_{episode:04d}.pt", episode, total_steps)
    plot_training(history, RUN_DIR / "training_dashboard.png")
    show_sample(gif_path, f"After {episode} training games")

### 5f. save the current result

Runs on completion, interruption, or failure. Checkpoints include every update so far; the CSV has finished games only.

In [ ]:
def save_training_result(status):
    save_checkpoint(model, RUN_DIR / "trained.pt", completed_episodes, total_steps)
    save_metrics(history, RUN_DIR / "training.csv")
    plot_training(history, RUN_DIR / "training_dashboard.png")
    summary = {"status": status, "completed_episodes": completed_episodes,
               "total_decisions": total_steps, "learning_updates": updates,
               "elapsed_seconds_including_periodic_demos": time.monotonic() - started}
    (RUN_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2))

### 5g. run the training games

The main loop. Each game prints its score. **Interrupt / Stop** saves the current model; then continue to section 6. For another experiment, rerun from 5a so the model, baseline, and results folder all reset together.

In [ ]:
if _training_started:
    raise RuntimeError("Start a new experiment by rerunning from section 5a, or use Run All.")
_training_started = True
train_env = make_env()
started = time.monotonic()
status = "completed"
try:
    for episode in range(1, EPISODES + 1):
        row = train_one_episode(train_env, episode)
        history.append(row)
        completed_episodes = episode
        recent_mean = np.mean([game["score"] for game in history[-25:]])
        print(f"Show {episode}/{EPISODES} | score {row['score']:.0f} | "
              f"recent mean {recent_mean:.1f} | exploration {row['exploration']:.0%}", flush=True)
        save_metrics(history, RUN_DIR / "training.csv")
        if episode % DEMO_EVERY == 0:
            save_progress_sample(episode)
except KeyboardInterrupt:
    status = "interrupted"
    print("Tour cut short. Saving your agent; continue to section 6 for results.")
except Exception:
    status = "failed"
    raise
finally:
    train_env.close()
    save_training_result(status)
print("Saved:", RUN_DIR)
if updates == 0:
    print("No learning updates yet: this run ended before a training batch was ready.")

## 6. did it learn? (the reviews are in)

Compare all five games under identical conditions. The featured GIF is the best of the five trained games.
It's a highlight reel, not evidence by itself. Five games is a small sample, not a peer-reviewed study.

### 6a. play the saved agent

Reload the checkpoint and play the same five seeds. This tests both the saved file and the agent's behavior.

In [ ]:
# Load the saved model so the evaluation also verifies checkpoint playback.
checkpoint = torch.load(RUN_DIR / "trained.pt", map_location="cpu", weights_only=True)
trained_model = DQN(checkpoint["n_actions"]).to(DEVICE)
trained_model.load_state_dict(checkpoint["model"])
print("Evaluating the saved trained network on the same five games. Moment of truth...", flush=True)
after = evaluate(trained_model, EVAL_SEEDS, RUN_DIR / "demos" / "final_best.gif", record_best=True)

### 6b. compare all five scores

A positive change means a higher average in this small test. A short run can get *worse*, and one good-looking clip is not proof. Report what actually happened, even if it's a flop era.

In [ ]:
comparison = {"baseline_kind": "untrained network", "before": baseline, "after": after,
              "evaluation_exploration": EVAL_EXPLORATION, "max_decisions_per_game": MAX_STEPS}
(RUN_DIR / "comparison.json").write_text(json.dumps(comparison, indent=2))
print(f"{'Game':>6} {'Before':>10} {'After':>10}")
for game, (before_score, after_score) in enumerate(zip(baseline["scores"], after["scores"]), 1):
    print(f"{game:>6} {before_score:>10.0f} {after_score:>10.0f}")
print(f"{'Mean':>6} {baseline['mean']:>10.1f} {after['mean']:>10.1f}")
change = after["mean"] - baseline["mean"]
print(f"Change in mean score: {change:+.1f}")
print("Verdict:", "Glow-up. Check the per-game scores before bragging." if change > 0 else
      "Flop era. Totally normal for short runs; write about why." if change < 0 else
      "Exactly the same. Statistically boring.")
print("Time-limited games before / after:", sum(baseline["time_limited"]), "/", sum(after["time_limited"]))
display(Image(filename=str(RUN_DIR / "training_dashboard.png")))
show_sample(RUN_DIR / "demos" / "final_best.gif", "Final saved agent (current era)")

## 7. save and explain your experiment

The next cell zips your settings, package versions, all scores, the training log, the plot, the SWIFT-MAN GIFs, and the checkpoints. In Colab, download it before the session ends.
To try another setting, change the three choices and Run All again; that starts from scratch.

Put the notebook, the plot, a few GIFs, `comparison.json`, and your explanation in your GitHub submission.
Checkpoints (`.pt`) are big; keep them local or attach them to a GitHub release.

In [ ]:
archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
try:
    from google.colab import files
except ImportError:
    display(FileLink(archive, result_html_prefix="Download your experiment: "))
else:
    files.download(archive)
print("Results folder:", RUN_DIR.resolve())

### your explanation (write this yourself, it's the part that gets graded)

- **My choices:** exploration = __, episodes = __, learning rate = __.
- **My prediction:** I expected these settings to __ because __.
- **What happened:** mean score before = __; after = __. In the gameplay, I noticed __.
- **One limitation:** __.
- **My next experiment:** change only __ to __ and keep the other two settings fixed.

The agent observes four screens, chooses a joystick action, and receives game rewards.
Lower training loss does not necessarily mean a higher game score. Report lack of progress if that is what happened. The glitter does not count as progress.

### implementation notes and sources

SWIFT-MAN is a cosmetic layer on the original course notebook (`pepealonso95/pacman-dqn`), built with Claude Code. The DQN, environment, rewards, and evaluation are unchanged; only recorded GIF frames are repainted.

This is a small, readable DQN experiment, not a reproduction of a large Atari benchmark.
Replay is deliberately small for laptops. Epsilon stays constant after warm-up. Checkpoints support playback;
rerunning starts a new experiment, rather than resuming the optimizer, replay memory, or an interrupted game.
GPU results can vary even with fixed seeds.

- [ALE installation and Gymnasium registration](https://ale.farama.org/getting-started/)
- [Gymnasium Atari preprocessing](https://gymnasium.farama.org/api/wrappers/misc_wrappers/#gymnasium.wrappers.AtariPreprocessing)
- [Gymnasium frame stacking](https://gymnasium.farama.org/api/wrappers/observation_wrappers/#gymnasium.wrappers.FrameStackObservation)
- [DQN paper: Human-level control through deep reinforcement learning](https://storage.googleapis.com/deepmind-media/dqn/DQNNaturePaper.pdf)